In [2]:
%pip install -q pandas chromadb numpy

In [3]:
from google.colab import files

uploaded = files.upload()

Saving faqs.csv to faqs.csv


In [4]:
import pandas as pd

df = pd.read_csv("faqs.csv")

print(f"Dataset loaded successfully: {len(df)} records")
display(df.head())

Dataset loaded successfully: 10 records


,id,question,answer,category
0,1,How can I reset my password?,Click Forgot Password on the login page and fo...,Account
1,2,How do I create a new account?,Open the registration page and complete the re...,Account
2,3,How can I change my email address?,Open your account settings and update your ema...,Account
3,4,How do I pay for my order?,Choose an available payment method during chec...,Payment
4,5,What should I do if my payment fails?,Check your payment details and try again or co...,Payment


## 1. Data Cleaning and Validation

We validate the required columns, remove records with missing values,
trim unnecessary whitespace, and remove duplicate questions.

In [5]:
required_columns = {"id", "question", "answer", "category"}

if not required_columns.issubset(df.columns):
    missing = required_columns - set(df.columns)
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(
    subset=["question", "answer", "category"]
).copy()

for column in ["question", "answer", "category"]:
    df[column] = df[column].astype(str).str.strip()

df = df[
    (df["question"] != "")
    & (df["answer"] != "")
    & (df["category"] != "")
].copy()

df = df.drop_duplicates(subset=["question"]).copy()
df["id"] = df["id"].astype(str)

if df.empty:
    raise ValueError("No valid FAQs were found.")

print(f"Valid FAQs after cleaning: {len(df)}")
display(df)

Valid FAQs after cleaning: 10


,id,question,answer,category
0,1,How can I reset my password?,Click Forgot Password on the login page and fo...,Account
1,2,How do I create a new account?,Open the registration page and complete the re...,Account
2,3,How can I change my email address?,Open your account settings and update your ema...,Account
3,4,How do I pay for my order?,Choose an available payment method during chec...,Payment
4,5,What should I do if my payment fails?,Check your payment details and try again or co...,Payment
5,6,How can I track my order?,Open your orders page and select the order you...,Delivery
6,7,How long does delivery take?,Delivery usually takes three to five business ...,Delivery
7,8,How can I cancel my order?,Open your orders page and cancel the order if ...,Orders
8,9,How do I contact customer support?,Send an email to support@example.com for assis...,Support
9,10,How can I get a refund?,Open your orders page and follow the refund re...,Orders


## 2. Generate Embeddings and Store Documents

We combine each FAQ question and answer into one document.
ChromaDB generates text embeddings and stores the documents
so we can search them by meaning.

In [6]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="faq_collection"
)

documents = (
    "Question: " + df["question"]
    + "\nAnswer: " + df["answer"]
).tolist()

metadatas = [
    {
        "question": row["question"],
        "answer": row["answer"],
        "category": row["category"],
    }
    for _, row in df.iterrows()
]

collection.add(
    ids=df["id"].tolist(),
    documents=documents,
    metadatas=metadatas,
)

print(f"Stored {collection.count()} FAQs in ChromaDB.")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 89.1MiB/s]


Stored 10 FAQs in ChromaDB.


## 3. Semantic Search

The search function accepts a natural-language question and retrieves
up to three relevant FAQs. Smaller distance values generally indicate
closer matches.

In [7]:
def search_faqs(query):
    query = query.strip()

    if not query:
        print("Please enter a search question.")
        return

    results = collection.query(
        query_texts=[query],
        n_results=min(3, collection.count()),
        include=["metadatas", "distances"],
    )

    print(f"Search results for: {query}")
    print("=" * 60)

    if not results["ids"] or not results["ids"][0]:
        print("No matching FAQs found.")
        return

    for rank, metadata in enumerate(
        results["metadatas"][0], start=1
    ):
        distance = results["distances"][0][rank - 1]

        print(f"\nResult {rank}")
        print(f"Category: {metadata['category']}")
        print(f"Question: {metadata['question']}")
        print(f"Answer: {metadata['answer']}")
        print(f"Distance: {distance:.4f}")

In [8]:
search_faqs("My payment failed. What should I do?")

Search results for: My payment failed. What should I do?

Result 1
Category: Payment
Question: What should I do if my payment fails?
Answer: Check your payment details and try again or contact your bank.
Distance: 0.3935

Result 2
Category: Orders
Question: How can I get a refund?
Answer: Open your orders page and follow the refund request instructions.
Distance: 1.1225

Result 3
Category: Support
Question: How do I contact customer support?
Answer: Send an email to support@example.com for assistance.
Distance: 1.1985


In [9]:
search_faqs("I forgot my login password.")

Search results for: I forgot my login password.

Result 1
Category: Account
Question: How can I reset my password?
Answer: Click Forgot Password on the login page and follow the instructions.
Distance: 0.5811

Result 2
Category: Account
Question: How can I change my email address?
Answer: Open your account settings and update your email address.
Distance: 1.3770

Result 3
Category: Support
Question: How do I contact customer support?
Answer: Send an email to support@example.com for assistance.
Distance: 1.4480


In [10]:
search_faqs("When will my package arrive?")

Search results for: When will my package arrive?

Result 1
Category: Delivery
Question: How long does delivery take?
Answer: Delivery usually takes three to five business days.
Distance: 1.0326

Result 2
Category: Delivery
Question: How can I track my order?
Answer: Open your orders page and select the order you want to track.
Distance: 1.5847

Result 3
Category: Payment
Question: How do I pay for my order?
Answer: Choose an available payment method during checkout.
Distance: 1.6421
